# 🐍 Daffa AI Coder — Multi-Language Training

Fine-tune **CodeT5-small** untuk generate kode di 11 bahasa pemrograman.

**Cara pakai:**
1. Runtime → Change runtime type → **T4 GPU**
2. Jalankan semua cell berurutan
3. Setelah selesai, model tersimpan di `./daffa-ai-coder-multilang`

In [ ]:
# 1. Install dependencies
!pip install -q transformers==4.46.3 datasets accelerate evaluate rouge-score sentencepiece peft

In [ ]:
# 2. Upload dataset (atau clone repo)
# Opsi A: clone repo kamu
!git clone https://github.com/daffadevhosting/fine-tuning-AI_coder.git
%cd fine-tuning-AI_coder

# Opsi B: jika dataset sudah di-update, upload file multi_lang_dataset.json ke /content/
# from google.colab import files
# files.upload()

In [ ]:
# 3. Cek GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Load dataset
import json
from pathlib import Path
from datasets import Dataset
from collections import Counter

DATA_PATH = Path("data/multi_lang_dataset.json")
if not DATA_PATH.exists():
    DATA_PATH = Path("/content/multi_lang_dataset.json")

with open(DATA_PATH, encoding="utf-8") as f:
    raw = json.load(f)

records = []
for item in raw:
    records.append({
        "input_text": f"Generate {item['language']} code: {item['instruction']}",
        "target_text": item["code"],
        "language": item["language"],
    })

full_ds = Dataset.from_list(records)
split = full_ds.train_test_split(test_size=0.12, seed=42)
train_ds = split["train"]
eval_ds = split["test"]

print(f"Total: {len(full_ds)} | Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print("Languages:", dict(Counter(r["language"] for r in records)))

In [ ]:
# 5. Load model & tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "Salesforce/codet5-small"  # ganti ke codet5-base jika VRAM cukup

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# 6. Preprocess
MAX_INPUT = 128
MAX_TARGET = 256

def preprocess(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length",
    )
    labels_ids = [
        [(t if t != tokenizer.pad_token_id else -100) for t in seq]
        for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = labels_ids
    return model_inputs

tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
tokenized_eval = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)
print("Tokenization done.")

In [ ]:
# 7. (Opsional) LoRA — hemat VRAM, cocok untuk codet5-base
USE_LORA = False  # set True jika pakai codet5-base atau VRAM terbatas

if USE_LORA:
    from peft import LoraConfig, get_peft_model, TaskType
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q", "v"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    print("Full fine-tuning (no LoRA)")

In [ ]:
# 8. Training
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir="./daffa-ai-coder-multilang",
    overwrite_output_dir=True,
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    generation_num_beams=2,
    fp16=True,
    logging_steps=10,
    report_to="none",
    dataloader_num_workers=2,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...")
train_result = trainer.train()
print("Metrics:", train_result.metrics)

In [ ]:
# 9. Save model
OUTPUT_DIR = "./daffa-ai-coder-multilang"
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

In [ ]:
# 10. Test generate
from transformers import pipeline

coder = pipeline(
    "text2text-generation",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    max_new_tokens=256,
    num_beams=3,
)

tests = [
    ("python", "Write a function to check if a number is prime"),
    ("javascript", "Write a function to debounce another function"),
    ("sql", "Write a SQL query to count orders per customer"),
    ("go", "Write a function to reverse a string"),
    ("java", "Write a method to calculate factorial"),
]

for lang, instr in tests:
    prompt = f"Generate {lang} code: {instr}"
    out = coder(prompt)[0]["generated_text"]
    print(f"\n{'='*50}")
    print(f"[{lang}] {instr}")
    print(out)

In [ ]:
# 11. (Opsional) Push ke Hugging Face Hub
# from huggingface_hub import notebook_login
# notebook_login()
#
# trainer.push_to_hub("your-username/daffa-ai-coder")
# print("Pushed to Hub!")

## Setelah training

1. Download folder `daffa-ai-coder-multilang` (klik kanan di file browser Colab → Download)
2. Atau push ke Hugging Face Hub (cell 11)
3. Update `app.py`:
```python
MODEL_PATH = "./daffa-ai-coder-multilang"  # atau username/repo-hf
```
4. Jalankan `python app.py` untuk demo Gradio